# Phase 12 — Streamlit Deployment

**Objective:** Package the trained dual-stream model into an interactive web application using Streamlit. The app accepts an uploaded image, runs it through both the spatial and frequency streams, returns a REAL/FAKE prediction with a confidence score, and displays the Grad-CAM heatmap overlay for explainability.

**This notebook does two things:**
1. Exports the model and all required assets from Kaggle to `/kaggle/working/`
2. Writes the complete `app.py` Streamlit application file ready for deployment

**Deployment target:** Streamlit Community Cloud (free) or Hugging Face Spaces

## Import Libraries

In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model

2026-06-03 20:09:39.432360: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780517379.627924      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780517379.682210      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780517380.131449      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780517380.131492      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780517380.131495      23 computation_placer.cc:177] computation placer alr

## Verify Model File Exists

In [2]:
MODEL_PATH = "/kaggle/input/models/jasminsultanashimu/p-10-final-model/keras/default/1/dual_stream_final.keras"
DEPLOY_DIR = "/kaggle/working/deploy"

os.makedirs(DEPLOY_DIR, exist_ok=True)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model not found at {MODEL_PATH}\n"
        "Run Phase 10 first, or copy dual_stream_final.keras to /kaggle/working/models/"
    )

size_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)
print(f"Model found : {MODEL_PATH}")
print(f"Size        : {size_mb:.1f} MB")


Model found : /kaggle/input/models/jasminsultanashimu/p-10-final-model/keras/default/1/dual_stream_final.keras
Size        : 60.6 MB


## Smoke-Test the Model on a Sample Image

Verifies that the saved model loads correctly and produces output in the expected range before writing the deployment app.

In [3]:
def dct_preprocess(image):
    image = image.astype(np.uint8)
    gray  = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    gray  = cv2.resize(gray, (128, 128))
    gray  = np.float32(gray)
    dct   = cv2.dct(gray)
    dct   = np.log(np.abs(dct) + 1)
    dct   = cv2.normalize(dct, None, 0, 1, cv2.NORM_MINMAX)
    dct   = np.stack([dct] * 3, axis=-1)
    return dct


model      = load_model(MODEL_PATH)
dummy_rgb  = np.random.rand(1, 128, 128, 3).astype(np.float32)
dummy_dct  = np.random.rand(1, 128, 128, 3).astype(np.float32)
test_pred  = model.predict([dummy_rgb, dummy_dct], verbose=0)

print(f"Model loaded successfully.")
print(f"Test prediction output : {test_pred[0][0]:.4f}  (should be in [0, 1])")

I0000 00:00:1780517394.557597      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1780517394.563933      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1780517403.955564      68 service.cc:152] XLA service 0x78f34c004320 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780517403.955624      68 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1780517403.955631      68 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1780517405.706457      68 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-06-03 20:10:11.509616: E external/local_xla/xla/stream_executor/cuda/c

Model loaded successfully.
Test prediction output : 1.0000  (should be in [0, 1])


## Copy Model to Deploy Directory

In [4]:
import shutil

shutil.copy(MODEL_PATH, os.path.join(DEPLOY_DIR, "dual_stream_final.keras"))
print("Model copied to deploy directory.")

Model copied to deploy directory.


## Write requirements.txt

In [5]:
requirements = """streamlit
tensorflow
opencv-python-headless
numpy
matplotlib
Pillow
"""

with open(os.path.join(DEPLOY_DIR, "requirements.txt"), "w") as f:
    f.write(requirements)

print("requirements.txt written.")

requirements.txt written.


## Write Streamlit Application — app.py

The application performs the following steps on every uploaded image:

| Step | Detail |
|---|---|
| Preprocessing | RGB normalization + DCT frequency transform |
| Inference | Dual-stream model forward pass |
| Prediction | Score → REAL / FAKE label + confidence |
| Explainability | Grad-CAM heatmap from spatial stream |
| Display | Original image, prediction, and heatmap overlay |

In [6]:
app_code = '''
import streamlit as st
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from PIL import Image
import matplotlib.pyplot as plt
import io


# ===== PAGE CONFIG =====
st.set_page_config(
    page_title="Deepfake Detector",
    page_icon="🔍",
    layout="wide"
)


# ===== MODEL LOADING =====
@st.cache_resource
def load_detector():
    return load_model("dual_stream_final.keras")


model = load_detector()


# ===== PREPROCESSING =====
def dct_preprocess(image):
    image = image.astype(np.uint8)
    gray  = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    gray  = cv2.resize(gray, (128, 128))
    gray  = np.float32(gray)
    dct   = cv2.dct(gray)
    dct   = np.log(np.abs(dct) + 1)
    dct   = cv2.normalize(dct, None, 0, 1, cv2.NORM_MINMAX)
    dct   = np.stack([dct] * 3, axis=-1)
    return dct


def prepare_inputs(pil_image):
    img_rgb  = np.array(pil_image.convert("RGB"))
    img_128  = cv2.resize(img_rgb, (128, 128))
    rgb_in   = img_128.astype(np.float32) / 255.0
    dct_in   = dct_preprocess(img_rgb)
    return img_128, rgb_in, dct_in


# ===== GRAD-CAM =====
def compute_gradcam(model, rgb_in, dct_in, last_conv_name="top_conv"):
    try:
        grad_model = Model(
            inputs=model.inputs,
            outputs=[model.get_layer(last_conv_name).output, model.output]
        )
        with tf.GradientTape() as tape:
            conv_out, preds = grad_model(
                [np.expand_dims(rgb_in, 0), np.expand_dims(dct_in, 0)]
            )
            score = preds[:, 0]

        grads        = tape.gradient(score, conv_out)
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
        conv_out     = conv_out[0]
        heatmap      = (conv_out @ pooled_grads[..., tf.newaxis]).numpy().squeeze()
        heatmap      = np.maximum(heatmap, 0)
        if heatmap.max() != 0:
            heatmap /= heatmap.max()
        return heatmap
    except Exception:
        return None


def overlay_heatmap(original, heatmap, alpha=0.4):
    heatmap_resized = cv2.resize(heatmap, (original.shape[1], original.shape[0]))
    colored         = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    colored         = cv2.cvtColor(colored, cv2.COLOR_BGR2RGB)
    blended         = cv2.addWeighted(original.astype(np.uint8), 1 - alpha, colored, alpha, 0)
    return blended


# ===== UI LAYOUT =====
st.title("🔍 Deepfake Image Detector")
st.markdown(
    "Upload an image to detect whether it is **REAL** or **AI-generated (FAKE)**.  "
    "The model uses a dual-stream architecture combining spatial and frequency-domain analysis."
)

st.divider()

upload = st.file_uploader("Upload Image", type=["jpg", "jpeg", "png"])

if upload is not None:
    pil_img = Image.open(upload)

    st.subheader("Uploaded Image")
    st.image(pil_img, width=300)

    with st.spinner("Analysing image..."):
        img_display, rgb_in, dct_in = prepare_inputs(pil_img)

        score = model.predict(
            [np.expand_dims(rgb_in, 0), np.expand_dims(dct_in, 0)], verbose=0
        )[0][0]

        label      = "FAKE" if score >= 0.5 else "REAL"
        confidence = score if score >= 0.5 else 1 - score

    st.divider()
    st.subheader("Prediction Result")

    col1, col2 = st.columns(2)

    with col1:
        if label == "FAKE":
            st.error(f"🚨 Prediction : **{label}**")
        else:
            st.success(f"✅ Prediction : **{label}**")
        st.metric("Confidence", f"{confidence * 100:.1f}%")
        st.metric("Raw Score (FAKE probability)", f"{score:.4f}")

    with col2:
        heatmap = compute_gradcam(model, rgb_in, dct_in)
        if heatmap is not None:
            overlay = overlay_heatmap(img_display, heatmap)
            st.image(overlay, caption="Grad-CAM — Spatial Attention Map", width=300)
        else:
            st.info("Grad-CAM visualization unavailable for this model configuration.")

    st.divider()
    st.subheader("Frequency Domain View")
    dct_vis = dct_in[:, :, 0]
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(dct_vis, cmap="inferno")
    ax.set_title("DCT Frequency Spectrum")
    ax.axis("off")
    buf = io.BytesIO()
    plt.savefig(buf, format="png", bbox_inches="tight", dpi=120)
    buf.seek(0)
    st.image(buf, width=300)
    plt.close()

else:
    st.info("Upload a JPG or PNG image to begin.")
'''

with open(os.path.join(DEPLOY_DIR, "app.py"), "w") as f:
    f.write(app_code)

print("app.py written successfully.")
print(f"Deploy directory contents: {os.listdir(DEPLOY_DIR)}")

app.py written successfully.
Deploy directory contents: ['requirements.txt', 'dual_stream_final.keras', 'app.py']


## List All Deployment Files

In [7]:
print("Files ready for deployment:")
for fname in os.listdir(DEPLOY_DIR):
    fpath = os.path.join(DEPLOY_DIR, fname)
    size  = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {fname:<35} {size:.2f} MB")

Files ready for deployment:
  requirements.txt                    0.00 MB
  dual_stream_final.keras             60.64 MB
  app.py                              0.00 MB


## Deployment Instructions

The three files in `/kaggle/working/deploy/` are everything needed to run the app.

**Option A — Streamlit Community Cloud (recommended)**

1. Push `app.py`, `requirements.txt`, and `dual_stream_final.keras` to a GitHub repository
2. Go to [share.streamlit.io](https://share.streamlit.io)
3. Connect the repository and set `app.py` as the entry point
4. Deploy — Streamlit installs dependencies and launches automatically

**Option B — Hugging Face Spaces**

1. Create a new Space with the Streamlit SDK selected
2. Upload `app.py`, `requirements.txt`, and the model file
3. The Space builds and launches automatically

**Option C — Local testing**

```bash
pip install streamlit tensorflow opencv-python-headless
streamlit run app.py
```

---
## Phase 12 Complete

| File | Purpose |
|---|---|
| `app.py` | Full Streamlit application |
| `requirements.txt` | Python dependencies |
| `dual_stream_final.keras` | Trained dual-stream model |

**App features:**
- Image upload (JPG / PNG)
- REAL / FAKE prediction with confidence score
- Grad-CAM spatial attention overlay
- DCT frequency spectrum visualization

---
## All 12 Phases Complete

## Download Phase 12 Outputs

Run this cell after the deployment files are written. This zip is the complete Streamlit deployment package.

In [8]:
import shutil, os

OUTPUT_FILES = [
    "/kaggle/working/deploy/app.py",
    "/kaggle/working/deploy/requirements.txt",
    "/kaggle/working/deploy/dual_stream_final.keras",
]

shutil.make_archive("/kaggle/working/phase_12_outputs", "zip", "/kaggle/working/deploy")

print("Files packaged:")
for f in OUTPUT_FILES:
    exists = os.path.exists(f)
    size   = round(os.path.getsize(f) / (1024 * 1024), 2) if exists else 0
    print(f"  {'OK' if exists else 'MISSING':<8} {os.path.basename(f):<45} {size} MB")

print()
print("Download: /kaggle/working/phase_12_outputs.zip")
print()
print("Upload the contents to GitHub or Hugging Face Spaces to deploy the app.")

Files packaged:
  OK       app.py                                        0.0 MB
  OK       requirements.txt                              0.0 MB
  OK       dual_stream_final.keras                       60.64 MB

Download: /kaggle/working/phase_12_outputs.zip

Upload the contents to GitHub or Hugging Face Spaces to deploy the app.
